# Verifier Failure Analysis

This notebook analyzes meaningful verifier outputs to identify strengths, weaknesses, gaps, and candidates for future rule refinement.

**Input:** `results/verifier_batch_meaningful.csv` (36 rows) and `results/verifier_batch_all.csv` (779 rows, for uncovered patterns)

**Source batch:** `src/verifier/run_verifier_batch.py` (first 50 train sentences)

Analysis only. No verifier logic, mapper, Stanza, or HTDB integration is modified here.

Observations are saved to `docs/verifier_failure_analysis_v1.md`.

## 1. Load Verifier Outputs

In [ ]:
import csv
from collections import Counter, defaultdict
from pathlib import Path

MEANINGFUL_PATH = Path("../results/verifier_batch_meaningful.csv")
ALL_PATH = Path("../results/verifier_batch_all.csv")


def load_csv(filepath):
    with open(filepath, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


meaningful = load_csv(MEANINGFUL_PATH)
all_rows = load_csv(ALL_PATH)

confirmed = [r for r in meaningful if r["decision_type"] == "confirmed"]
ambiguous = [r for r in meaningful if r["decision_type"] == "ambiguous"]
no_decision = [r for r in all_rows if r["decision_type"] == "no_decision"]
uncovered_with_case = [r for r in no_decision if r["case_marker"]]

print(f"Meaningful rows: {len(meaningful)}")
print(f"  confirmed: {len(confirmed)}")
print(f"  ambiguous: {len(ambiguous)}")
print(f"All batch tokens: {len(all_rows)}")
print(f"no_decision tokens with a case marker: {len(uncovered_with_case)}")

## 2. Helper: Display Tables and Examples

In [ ]:
TABLE_COLS = [
    "sent_id",
    "token_form",
    "deprel",
    "case_marker",
    "rule_id",
    "decision_type",
    "karaka_candidates",
]


def print_table(rows, extra_cols=None):
    """Print a simple text table from verifier rows."""
    cols = list(TABLE_COLS)
    if extra_cols:
        cols = cols + list(extra_cols)

    if not rows:
        print("(no rows)")
        return

    widths = {col: max(len(col), *(len(str(r.get(col, ""))) for r in rows)) for col in cols}
    header = "  ".join(col.ljust(widths[col]) for col in cols)
    print(header)
    print("-" * len(header))
    for row in rows:
        print("  ".join(str(row.get(col, "")).ljust(widths[col]) for col in cols))


def show_sentences(rows, label_col="assessment"):
    """Print example sentences with an optional assessment label."""
    for i, row in enumerate(rows, start=1):
        label = row.get(label_col, "")
        prefix = f"[{label}] " if label else ""
        print(f"Example {i} {prefix}({row['sent_id']}, {row['token_form']}, {row['rule_id']})")
        print(f"  {row['sentence_text']}")
        print()

## 3. Strongest Confirmed Examples

Manual assessment of the 19 `confirmed` rows. Strength is based on sentence context in the batch output, not gold Karaka labels.

| Strength | Criterion used in this notebook |
|----------|----------------------------------|
| **Strong** | Clear agent (R1), clear spatial container/surface (R2/R3) |
| **Moderate** | Plausible locus reading, but semantic subtype is less clear |
| **Weak** | Rule fires, but reading is extent, time, manner, or abstract rather than core location |

In [ ]:
# Manual strength labels for confirmed rows (sent_id + token_form keys).
CONFIRMED_STRENGTH = {
    ("train-s2", "शाहजेहन"): "strong",
    ("train-s11", "कोरिया"): "strong",
    ("train-s50", "राजाओं"): "strong",
    ("train-s4", "हॉल"): "strong",
    ("train-s12", "क्षेत्र"): "strong",
    ("train-s19", "संग्रहालय"): "strong",
    ("train-s32", "चौक"): "strong",
    ("train-s33", "गलियों"): "strong",
    ("train-s36", "झील"): "strong",
    ("train-s15", "हिल्स"): "strong",
    ("train-s46", "शहर"): "moderate",
    ("train-s46", "पत्थरों"): "moderate",
    ("train-s15", "एकड़"): "weak",
    ("train-s26", "हैक्टेयर"): "weak",
    ("train-s45", "वर्ष"): "weak",
    ("train-s48", "शताब्दी"): "weak",
    ("train-s49", "शताब्दी"): "weak",
    ("train-s34", "अंदाज"): "weak",
    ("train-s50", "शान"): "weak",
}

for row in confirmed:
    key = (row["sent_id"], row["token_form"])
    row["strength"] = CONFIRMED_STRENGTH.get(key, "unlabeled")

strength_counts = Counter(row["strength"] for row in confirmed)
print("Confirmed strength counts:")
for strength, count in sorted(strength_counts.items()):
    print(f"  {strength}: {count}")
print()

strong_confirmed = [r for r in confirmed if r["strength"] == "strong"]
print_table(strong_confirmed, extra_cols=["strength"])
print()
show_sentences(strong_confirmed, label_col="strength")

## 4. Ambiguous Examples: Genuinely Ambiguous vs Potentially Resolvable

All 17 ambiguous rows come from R4 or R5. Each row is classified using sentence context from the batch CSV.

| Category | Meaning in this notebook |
|----------|--------------------------|
| **Genuine ambiguity** | Karaṇa vs Apādāna (R4) or Karma vs Sampradāna (R5) remains plausible even with light context; or the hit is primarily a manner reading not well captured by the current candidate pair |
| **Potentially resolvable** | Additional context (verb lemma, frame, or fixed construction) may support one Karaka over the other in a later rule version |

In [ ]:
AMBIGUOUS_ASSESSMENT = {
    ("train-s10", "रूप"): ("genuine", "Manner adverbial (मुख्य रूप से); not a clear Karaṇa/Apādāna choice"),
    ("train-s16", "चित्रकला"): ("resolvable", "Material decoration frame (सज्जित); instrument/material Karaṇa likely"),
    ("train-s17", "तरह"): ("genuine", "Manner adverbial (इस तरह से)"),
    ("train-s17", "रूप"): ("genuine", "Manner adverbial (जीवंत रूप से)"),
    ("train-s20", "हिस्सों"): ("resolvable", "Collection source frame (एकत्रित); Apādāna likely"),
    ("train-s23", "जिलों"): ("resolvable", "Collection source frame (एकत्रित); Apādāna likely"),
    ("train-s25", "झील"): ("resolvable", "Adjacency frame (से लगी); separation/source reading"),
    ("train-s29", "रूप"): ("genuine", "Manner adverbial (मूल रूप से)"),
    ("train-s35", "झील"): ("resolvable", "Separation frame (अलग); Apādāna likely"),
    ("train-s35", "ओवरब्रिज"): ("resolvable", "Separation frame (अलग); Apādāna likely"),
    ("train-s41", "मुंबई"): ("resolvable", "Route origin frame (… से … जाने वाली); Apādāna likely"),
    ("train-s6", "लोगों"): ("genuine", "आमंत्रित: recipient (Sampradāna) vs theme (Karma) both plausible"),
    ("train-s19", "पुस्तकालय"): ("resolvable", "Passive/experiencer frame (देखा जाता है); patient Karma likely"),
    ("train-s23", "नमूनों"): ("resolvable", "Placement frame (रखा गया); patient Karma likely"),
    ("train-s27", "प्राणियों"): ("resolvable", "Perception frame (देखने); patient Karma likely"),
    ("train-s47", "चमक"): ("genuine", "Abstract object/experiencer; Karma vs Sampradāna unclear"),
    ("train-s48", "वैभव"): ("resolvable", "Expression frame (बयाँ करती हैं); direct object Karma likely"),
}

for row in ambiguous:
    key = (row["sent_id"], row["token_form"])
    category, note = AMBIGUOUS_ASSESSMENT[key]
    row["category"] = category
    row["note"] = note

amb_counts = Counter(row["category"] for row in ambiguous)
print("Ambiguous row categories:")
for category, count in sorted(amb_counts.items()):
    print(f"  {category}: {count}")
print()

by_rule = defaultdict(list)
for row in ambiguous:
    by_rule[row["rule_id"]].append(row)

for rule_id in sorted(by_rule):
    print(f"=== {rule_id} ambiguous breakdown ===")
    print_table(by_rule[rule_id], extra_cols=["category", "note"])
    print()

In [ ]:
genuine_amb = [r for r in ambiguous if r["category"] == "genuine"]
resolvable_amb = [r for r in ambiguous if r["category"] == "resolvable"]

print("Genuinely ambiguous examples:")
show_sentences(genuine_amb, label_col="note")

print("Potentially resolvable examples:")
show_sentences(resolvable_amb, label_col="note")

## 5. Rules That Appear Overly Broad

Summary based on confirmed and ambiguous rows in the meaningful CSV.

In [ ]:
overbroad = [
    {
        "rule_id": "R2",
        "issue": "Maps every obl+में hit to confirmed Adhikaraṇa",
        "weak_confirmed_count": sum(1 for r in confirmed if r["rule_id"] == "R2" and r["strength"] == "weak"),
        "r2_total": sum(1 for r in confirmed if r["rule_id"] == "R2"),
        "examples": "एकड़, हैक्टेयर (extent); वर्ष, शताब्दी (time); अंदाज (manner); शान (abstract)",
    },
    {
        "rule_id": "R4",
        "issue": "Single rule covers manner, source, separation, and material readings",
        "manner_like_count": sum(
            1 for r in ambiguous
            if r["rule_id"] == "R4" and r["category"] == "genuine"
        ),
        "r4_total": sum(1 for r in ambiguous if r["rule_id"] == "R4"),
        "examples": "रूप से, तरह से (manner); shared Karaṇa|Apādāna label either way",
    },
    {
        "rule_id": "R5",
        "issue": "obj/iobj+को always ambiguous even when verb frame suggests Karma",
        "resolvable_count": sum(
            1 for r in ambiguous
            if r["rule_id"] == "R5" and r["category"] == "resolvable"
        ),
        "r5_total": sum(1 for r in ambiguous if r["rule_id"] == "R5"),
        "examples": "देखने, रखा गया, बयाँ करती (patient-like frames)",
    },
]

print(f"{'Rule':<6} {'Issue'}")
print("-" * 70)
for item in overbroad:
    print(f"{item['rule_id']:<6} {item['issue']}")
    for k, v in item.items():
        if k not in ("rule_id", "issue"):
            print(f"         {k}: {v}")
    print()

## 6. Constructions Not Covered by Current Rules

From `verifier_batch_all.csv`: tokens with a case marker that still received `no_decision` (v1 rules did not apply).

In [ ]:
uncovered_pairs = Counter((r["deprel"], r["case_marker"]) for r in uncovered_with_case)

print(f"Uncovered tokens with case markers: {len(uncovered_with_case)}")
print(f"Unique (deprel, case_marker) pairs: {len(uncovered_pairs)}")
print()
print(f"{'Deprel':<14} {'Case':<8} {'Count':>6}  Example token / sent_id")
print("-" * 60)

uncovered_examples = []
for (deprel, case_marker), count in uncovered_pairs.most_common():
    example = next(
        r for r in uncovered_with_case
        if r["deprel"] == deprel and r["case_marker"] == case_marker
    )
    uncovered_examples.append({
        "deprel": deprel,
        "case_marker": case_marker,
        "count": str(count),
        "example_token": example["token_form"],
        "sent_id": example["sent_id"],
        "sentence_text": example["sentence_text"],
    })
    print(
        f"{deprel:<14} {case_marker:<8} {count:>6}  "
        f"{example['token_form']} / {example['sent_id']}"
    )

print()
print("Sample uncovered sentences (top pairs):")
for item in uncovered_examples[:8]:
    print(f"  {item['deprel']}+{item['case_marker']} ({item['example_token']}, {item['sent_id']})")
    print(f"    {item['sentence_text']}")
    print()

In [ ]:
# Additional no_decision patterns in the batch (no case marker on parent).
bare_nsubj = [r for r in no_decision if r["deprel"] == "nsubj" and not r["case_marker"]]
print(f"Bare nsubj tokens (no case child): {len(bare_nsubj)}")
print("v1 has no rule for nsubj without ने.")
print()
for row in bare_nsubj[:5]:
    print(f"  {row['sent_id']} | {row['token_form']} | {row['sentence_text'][:70]}")

## 7. Potential Future Rule Refinements

Hypotheses for a later rule version. Not implemented here.

In [ ]:
refinements = [
    {
        "target": "R2 subtypes",
        "motivation": f"{sum(1 for r in confirmed if r['rule_id']=='R2' and r['strength']=='weak')} of 15 R2 hits labeled weak (extent/time/manner)",
        "idea": "Split spatial location from temporal/extent/manner में frames before assigning Adhikaraṇa",
    },
    {
        "target": "R4 manner filter",
        "motivation": f"{sum(1 for r in ambiguous if r['rule_id']=='R4' and r['category']=='genuine')} R4 hits are manner adverbials (रूप/तरह से)",
        "idea": "Return no_decision or a separate non-Karaka manner label instead of Karaṇa|Apādāna",
    },
    {
        "target": "R4 verb-frame splits",
        "motivation": f"{sum(1 for r in ambiguous if r['rule_id']=='R4' and r['category']=='resolvable')} R4 hits look source/separation resolvable",
        "idea": "Use verb lemmas (एकत्रित, अलग, सज्जित, जाने वाली) to choose Apādāna vs Karaṇa",
    },
    {
        "target": "R5 verb-frame splits",
        "motivation": f"{sum(1 for r in ambiguous if r['rule_id']=='R5' and r['category']=='resolvable')} of 6 R5 hits labeled resolvable",
        "idea": "Use perception/placement/expression frames to confirm Karma; keep invitation frames ambiguous",
    },
    {
        "target": "Genitive and nmod coverage",
        "motivation": f"nmod+के/का/की account for {uncovered_pairs[('nmod','के')] + uncovered_pairs[('nmod','का')] + uncovered_pairs[('nmod','की')]} uncovered hits in batch",
        "idea": "Treat genitive case as NP-linking (no Karaka), as specified in rule spec exclusions",
    },
    {
        "target": "obl+को / temporal obl",
        "motivation": "obl+को and obl+तक appear in batch with no_decision",
        "idea": "Add dedicated rules or explicit no_decision reasons for dative obliques and endpoint markers",
    },
    {
        "target": "Bare nsubj",
        "motivation": f"{len(bare_nsubj)} bare nsubj tokens in batch; v1 only covers nsubj+ने",
        "idea": "Keep no_decision or add corrected rules once mapper hypotheses exist",
    },
]

print(f"{'Target':<24} {'Motivation'}")
print("-" * 80)
for item in refinements:
    print(f"{item['target']:<24} {item['motivation']}")
    print(f"{'':24} Idea: {item['idea']}")
    print()

## Summary

This notebook classified all 36 meaningful verifier rows and surveyed uncovered `(deprel, case_marker)` pairs in the full 779-token batch.

Key counts (from manual labels above):

- **Strong confirmed:** 10 of 19 confirmed rows
- **Moderate confirmed:** 2 rows
- **Weak confirmed (R2):** 7 rows where में marks extent, time, manner, or abstract locus
- **Genuinely ambiguous:** 6 of 17 ambiguous rows
- **Potentially resolvable ambiguous:** 11 of 17 ambiguous rows
- **Uncovered with case marker:** 73 tokens in the 50-sentence batch

Full observations: `docs/verifier_failure_analysis_v1.md`